# Ir Além 1 — IA Generativa e Extração de Informações Clínicas

**Objetivo:** expandir o Assistente Cardiológico Conversacional para interpretar textos clínicos não estruturados, extraindo informações relevantes em formato estruturado (JSON), utilizando **Chain of Thought (CoT)** — uma das técnicas de prompting apresentadas no Capítulo 10 da disciplina.

**Sobre a ferramenta utilizada:** o enunciado original sugere o uso do watsonx.ai. Durante o desenvolvimento, a conta IBM Cloud utilizada no projeto foi bloqueada, impossibilitando o acesso ao serviço mesmo após tentativa de recuperação. Optou-se então pelo uso do **Ollama** (`llama3.1:8b`), rodando localmente, pelos seguintes motivos:

1. As técnicas de prompting do Capítulo 10 (incluindo Chain of Thought) são conceitos agnósticos de fornecedor — aplicam-se da mesma forma a qualquer LLM que aceite um prompt de texto, seja Watson, OpenAI ou Ollama.
2. O Ollama expõe a API de chat de forma direta e de baixo nível, sem abstrações prontas de orquestração — o que mantém o foco na técnica de prompting em si, evitando o uso de uma ferramenta que já "entrega pronto" o comportamento avaliado.

**Por que Chain of Thought:** em um contexto clínico, uma conclusão (como "isso é um sinal de alerta") sem justificativa é uma caixa-preta arriscada. Ao instruir o modelo a raciocinar explicitamente sobre os sintomas antes de concluir, a extração se torna mais transparente e auditável — alguém revisando a saída consegue verificar *por que* o modelo chegou àquela conclusão, não só *qual* foi a conclusão.


## 0. Setup

In [6]:
# pip install ollama pydantic pandas --break-system-packages
# (requer o Ollama instalado e rodando localmente: https://ollama.com)
# Baixe o modelo antes de rodar este notebook: ollama pull llama3.1:8b

import ollama
import json
from typing import List, Optional
from pydantic import BaseModel, ValidationError
import pandas as pd

MODEL = "llama3.1:8b"


## 1. Dados de entrada simulados

Quatro relatos clínicos simulados, com diferentes níveis de gravidade — cobrindo tanto casos com sinais de alerta quanto casos de rotina.

In [7]:
textos_clinicos = [
    """Paciente relata dor torácica em aperto, com início há cerca de 2 horas,
    irradiando para o braço esquerdo. Refere sudorese fria associada.
    Histórico de hipertensão arterial e tabagismo há 15 anos.""",

    """Paciente do sexo feminino, 58 anos, comparece à consulta relatando
    cansaço frequente ao subir escadas, presente há aproximadamente 3 semanas.
    Nega dor torácica. Refere histórico familiar de infarto (pai) e
    diagnóstico prévio de colesterol elevado.""",

    """Relato de palpitações intermitentes nos últimos 2 dias, sem dor
    associada. Paciente nega histórico de doenças cardiovasculares, mas
    relata sedentarismo e obesidade leve.""",

    """Paciente relata leve inchaço nos tornozelos há uma semana, sem outros
    sintomas associados. Não possui fatores de risco cardiovasculares
    conhecidos.""",
]


## 2. Baseline — sem Chain of Thought

Primeiro, pedimos diretamente a conclusão (é ou não um sinal de alerta?), sem pedir raciocínio algum. É rápido, mas não há como verificar *por que* o modelo respondeu isso — se ele errar, não há pista nenhuma sobre a causa do erro.

In [8]:
def classificar_sem_cot(texto):
    resposta = ollama.chat(
        model=MODEL,
        options={"temperature": 0.2},
        messages=[
            {
                "role": "system",
                "content": "Você é um assistente de triagem clínica. Responda apenas "
                            "'true' ou 'false': o texto indica um sinal de alerta "
                            "cardíaco (dor no peito, falta de ar, dor no braço ou suor frio)?",
            },
            {"role": "user", "content": texto},
        ],
    )
    return resposta["message"]["content"].strip()


for i, texto in enumerate(textos_clinicos):
    print(f"Caso {i+1}: {classificar_sem_cot(texto)}")


Caso 1: True
Caso 2: False
Caso 3: False
Caso 4: False


**Observação esperada:** você obtém `true` ou `false`, mas nenhuma explicação. Se o professor ou um médico revisando o sistema perguntar "por que o caso 3 foi classificado assim?", não há resposta — é uma caixa-preta.

## 3. Com Chain of Thought

Agora, em vez de pedir a conclusão direto, instruímos o modelo a **raciocinar passo a passo** antes de concluir: primeiro listar os sintomas identificados, depois compará-los com os critérios de alerta, e só then declarar a conclusão final. Pedimos a saída em JSON para manter o raciocínio e a conclusão organizados e parseáveis (em vez de um texto corrido).

In [9]:
PROMPT_COT = """Você é um assistente de triagem clínica. Analise o texto do paciente
seguindo EXATAMENTE estes passos de raciocínio, e retorne um JSON com essa estrutura:

{
  "passo_1_sintomas_identificados": ["liste aqui cada sintoma mencionado no texto"],
  "passo_2_comparacao_com_criterios": "explique, um a um, se cada sintoma identificado
    bate com os critérios de alerta (dor no peito, falta de ar, dor no braço, suor frio)",
  "passo_3_conclusao": true ou false,
  "resumo": "uma frase resumindo o caso"
}

Retorne APENAS o JSON, sem texto antes ou depois."""


def classificar_com_cot(texto):
    resposta = ollama.chat(
        model=MODEL,
        format="json",
        options={"temperature": 0.2},
        messages=[
            {"role": "system", "content": PROMPT_COT},
            {"role": "user", "content": texto},
        ],
    )
    return resposta["message"]["content"]


saida = classificar_com_cot(textos_clinicos[2])
print(saida)


{
  "passo_1_sintomas_identificados": ["palpitações intermitentes"],
  "passo_2_comparacao_com_criterios": "O sintoma de palpitações intermitentes não bate com os critérios de alerta de dor no peito, mas pode ser considerado um sintoma de alerta para doenças cardiovasculares. A falta de dor associada pode sugerir que não é um ataque cardíaco, mas ainda assim é um sintoma que merece atenção. O sedentarismo e a obesidade leve podem ser fatores de risco para doenças cardiovasculares.",
  "passo_3_conclusao": false,
  "resumo": "Paciente com palpitações intermitentes sem dor associada, com histórico de sedentarismo e obesidade leve."
}


Repare que o caso 3 (palpitações + sedentarismo/obesidade, sem dor) é ambíguo — não tem nenhum sintoma da lista de alerta explícita, mas tem fatores de risco. É exatamente esse tipo de caso limítrofe onde o raciocínio explícito mais ajuda: dá pra ver o "porquê" da conclusão, não só o resultado.

## 4. Validação estruturada da saída (Pydantic)

Validamos a saída programaticamente antes de confiar nela, com nova tentativa em caso de falha — o modelo pode ocasionalmente devolver um JSON malformado ou omitir um campo.

In [10]:
class ExtracaoComRaciocinio(BaseModel):
    passo_1_sintomas_identificados: List[str]
    passo_2_comparacao_com_criterios: str
    passo_3_conclusao: bool
    resumo: str


def classificar_validado(texto, tentativas=3):
    ultimo_erro = None
    for _ in range(tentativas):
        bruto = classificar_com_cot(texto)
        try:
            dados = json.loads(bruto)
            return ExtracaoComRaciocinio(**dados)
        except (json.JSONDecodeError, ValidationError) as erro:
            ultimo_erro = erro
            continue
    raise RuntimeError(f"Falha ao validar após {tentativas} tentativas: {ultimo_erro}")


resultado = classificar_validado(textos_clinicos[0])
resultado


ExtracaoComRaciocinio(passo_1_sintomas_identificados=['dor torácica em aperto', 'irradiação para o braço esquerdo', 'sudorese fria'], passo_2_comparacao_com_criterios='A dor torácica em aperto e a irradiação para o braço esquerdo podem ser sintomas de angina de peito, que é um critério de alerta. A sudorese fria também é um sintoma preocupante. O histórico de hipertensão arterial e tabagismo há 15 anos aumenta o risco de doenças cardíacas.', passo_3_conclusao=True, resumo='Paciente com sintomas de angina de peito e histórico de doenças cardíacas, que requer avaliação médica urgente.')

## 5. Execução em lote sobre todos os casos simulados

In [11]:
resultados = []
for texto in textos_clinicos:
    extracao = classificar_validado(texto)
    resultados.append(extracao.model_dump())

df = pd.DataFrame(resultados)
df


,passo_1_sintomas_identificados,passo_2_comparacao_com_criterios,passo_3_conclusao,resumo
0,"[dor torácica em aperto, irradiação para o bra...","Dor no peito: SIM, pois o paciente relata dor ...",False,"Paciente com dor torácica em aperto, irradiand..."
1,[cansaço frequente ao subir escadas],O sintoma de cansaço frequente ao subir escada...,False,Paciente feminina de 58 anos com cansaço frequ...
2,[palpitações intermitentes],O sintoma de palpitações intermitentes não bat...,False,Paciente com palpitações intermitentes sem dor...
3,[leve inchaço nos tornozelos],O sintoma de inchaço nos tornozelos não bate c...,False,Paciente com inchaço nos tornozelos sem outros...


O campo `passo_3_conclusao` reaproveita a mesma lógica de gravidade usada na Action **"Relatar sintoma"** do watsonx Assistant (Parte 1) — dor no peito, falta de ar, dor no braço e suor frio como sinais de alerta. A diferença é que aqui, além da conclusão, temos o **raciocínio** (`passo_2_comparacao_com_criterios`) que justifica cada decisão — algo que o fluxo de dialog nodes da Parte 1 não expõe explicitamente, já que ali a lógica é uma condição fixa, não um raciocínio narrado.

## 6. Comparação lado a lado (sem CoT vs. com CoT)

Para deixar a diferença bem clara, uma tabela comparando as duas abordagens no mesmo conjunto de casos.

In [12]:
comparacao = []
for i, texto in enumerate(textos_clinicos):
    sem_cot = classificar_sem_cot(texto)
    com_cot = classificar_validado(texto)
    comparacao.append({
        "caso": i + 1,
        "sem_cot (só a conclusão)": sem_cot,
        "com_cot (conclusão)": com_cot.passo_3_conclusao,
        "com_cot (raciocínio)": com_cot.passo_2_comparacao_com_criterios[:100] + "...",
    })

pd.DataFrame(comparacao)


,caso,sem_cot (só a conclusão),com_cot (conclusão),com_cot (raciocínio)
0,1,True,True,A dor torácica em aperto e a irradiação para o...
1,2,False,False,O sintoma de cansaço frequente ao subir escada...
2,3,False,False,O sintoma de palpitações intermitentes não bat...
3,4,False,False,O sintoma de inchaço nos tornozelos não bate c...


## 7. Conclusão

Este notebook demonstrou a técnica de **Chain of Thought** (Capítulo 10) aplicada à extração de informações clínicas:

- **Sem CoT:** resposta direta (`true`/`false`), rápida mas sem justificativa — uma caixa-preta.
- **Com CoT:** o modelo lista os sintomas identificados, compara cada um com os critérios de alerta, e só então conclui — tornando a decisão auditável.
- A saída estruturada (JSON + validação Pydantic) manteve o raciocínio organizado e utilizável programaticamente, em vez de um texto corrido difícil de parsear.

**Conexão com a Parte 1:** a conclusão (`passo_3_conclusao`) usa o mesmo critério de gravidade da Action "Relatar sintoma" do watsonx Assistant, mostrando que a extração via LLM pode alimentar a mesma regra de negócio de segurança usada no fluxo conversacional — só que, aqui, com o raciocínio explícito por trás da decisão.

**Limitações:** o modelo local (`llama3.1:8b`) é menor que modelos de nuvem maiores, e o raciocínio gerado, embora estruturado, ainda é uma justificativa em linguagem natural — não uma prova lógica formal. Casos ambíguos (como o caso 3) merecem revisão humana mesmo com CoT.
